# Lab 01: Univariate Linear Regression, Cost Geometry & Computational Graph Traces

[Back to Session Overview](../README.md) · [Module 02 Index](../../README.md)

---

## Overview

Before tackling multi-feature regression on tabular datasets, it is essential to build geometric intuition for what a linear model and its squared-error cost function look like.

In this notebook, you will:
1. **Model Univariate Data:** Fit $h_\theta(x) = \theta_0 + \theta_1 x$ to the toy house-price dataset and the identity benchmark $(1,1), (2,2), (3,3)$.
2. **Visualize the Cost Surface $J(\theta_1)$ and $J(\theta_0, \theta_1)$:** Inspect the 1D parabola, 3D convex bowl, and 2D contour lines.
3. **Trace Gradient Descent Dynamics:** Compare slow, optimal, and divergent learning rates $\alpha$, verifying convexity (no spurious local minima).
4. **Verify the Computational Graph Hand-Trace:** Reproduce the exact single-sample update on $x = 6.7, y = 9.1, w = -0.34, b = 0.04, \eta = 0.01$.
5. **Solve Conceptual Diagnostic Quizzes:** Confirm key properties of learning rate selection and convexity.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

---

## 1. Univariate House Price Dataset & Hypothesis

We begin with the small house-area and price table from the lecture exercises:

| Area ($\mathrm{ft}^2$) | Price (\$k) |
| :---: | :---: |
| 500 | 135 |
| 852 | 178 |
| 1416 | 232 |
| 1534 | 315 |
| 2140 | 460 |

And for illustrating $J(\theta_1)$ when $\theta_0 = 0$, we use the three identity points $(1, 1), (2, 2), (3, 3)$.


In [ ]:
areas = np.array([500.0, 852.0, 1416.0, 1534.0, 2140.0])
prices = np.array([135.0, 178.0, 232.0, 315.0, 460.0])

# Closed-form check for intuition (mean-centered slope and intercept)
x_mean, y_mean = areas.mean(), prices.mean()
theta1_opt = np.sum((areas - x_mean) * (prices - y_mean)) / np.sum((areas - x_mean) ** 2)
theta0_opt = y_mean - theta1_opt * x_mean
print(f"Best fit line: price = {theta0_opt:.2f} + {theta1_opt:.4f} * area")

plt.figure(figsize=(8, 4.5))
plt.scatter(areas, prices, color="#2563eb", s=70, label="Training samples")
x_grid = np.linspace(400, 2250, 100)
plt.plot(x_grid, theta0_opt + theta1_opt * x_grid, color="#dc2626", lw=2, label="Linear fit")
plt.xlabel("Area (ft^2)")
plt.ylabel("Price ($1000s)")
plt.title("Univariate House Price Prediction")
plt.legend()
plt.tight_layout()
plt.show()

---

## 2. Cost Function $J(\theta_1)$ and Convex Bowl $J(\theta_0, \theta_1)$

With $\theta_0 = 0$ and $h_\theta(x) = \theta_1 x$, the half-MSE cost is:

$$J(\theta_1) = \frac{1}{2m} \sum_{i=1}^{m} \bigl(\theta_1 x^{(i)} - y^{(i)}\bigr)^2$$

On $(1, 1), (2, 2), (3, 3)$, evaluating $J(\theta_1)$ across $\theta_1 \in [-0.5, 2.5]$ traces a smooth parabola centered at $\theta_1^* = 1.0$.


In [ ]:
x_id = np.array([1.0, 2.0, 3.0])
y_id = np.array([1.0, 2.0, 3.0])


def compute_cost_1d(theta1: float, x: np.ndarray, y: np.ndarray) -> float:
    m = len(x)
    preds = theta1 * x
    return float(np.sum((preds - y) ** 2) / (2.0 * m))


theta1_vals = np.linspace(-0.5, 2.5, 121)
j_vals = [compute_cost_1d(t, x_id, y_id) for t in theta1_vals]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for t1, col in [(0.0, "#64748b"), (0.5, "#f59e0b"), (1.0, "#16a34a"), (1.5, "#dc2626")]:
    axes[0].plot(x_id, t1 * x_id, color=col, lw=2, label=f"theta1 = {t1}")
axes[0].scatter(x_id, y_id, color="black", marker="x", s=80, zorder=5, label="Data")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_title("Hypotheses h(x) = theta1 * x")
axes[0].legend()

axes[1].plot(theta1_vals, j_vals, color="#2563eb", lw=2)
axes[1].scatter([1.0], [0.0], color="#16a34a", s=80, zorder=5, label="Global min (1.0, 0.0)")
axes[1].set_xlabel("theta1")
axes[1].set_ylabel("J(theta1)")
axes[1].set_title("1D Cost Parabola J(theta1)")
axes[1].legend()

plt.tight_layout()
plt.show()

Extending to both parameters $(\theta_0, \theta_1)$ gives a convex 2D bowl $J(\theta_0, \theta_1)$. Notice that every contour ellipse encloses a single global minimum—there are no local minima traps.


In [ ]:
def compute_cost_2d(theta0: float, theta1: float, x: np.ndarray, y: np.ndarray) -> float:
    m = len(x)
    preds = theta0 + theta1 * x
    return float(np.sum((preds - y) ** 2) / (2.0 * m))


t0_grid = np.linspace(-2.0, 2.0, 80)
t1_grid = np.linspace(-0.5, 2.5, 80)
T0, T1 = np.meshgrid(t0_grid, t1_grid)
J_surface = np.zeros_like(T0)
for r in range(T0.shape[0]):
    for c in range(T0.shape[1]):
        J_surface[r, c] = compute_cost_2d(T0[r, c], T1[r, c], x_id, y_id)


def run_batch_gd(
    x: np.ndarray, y: np.ndarray, theta0: float, theta1: float, alpha: float, steps: int
) -> tuple[list[tuple[float, float]], list[float]]:
    m = len(x)
    path = [(theta0, theta1)]
    costs = [compute_cost_2d(theta0, theta1, x, y)]
    for _ in range(steps):
        err = (theta0 + theta1 * x) - y
        grad0 = float(np.sum(err) / m)
        grad1 = float(np.sum(err * x) / m)
        # Simultaneous update
        theta0 = theta0 - alpha * grad0
        theta1 = theta1 - alpha * grad1
        path.append((theta0, theta1))
        costs.append(compute_cost_2d(theta0, theta1, x, y))
    return path, costs


path_ok, costs_ok = run_batch_gd(x_id, y_id, -1.5, 2.2, alpha=0.1, steps=40)
path_fast, costs_fast = run_batch_gd(x_id, y_id, -1.5, 2.2, alpha=0.35, steps=40)
path_div, costs_div = run_batch_gd(x_id, y_id, -1.5, 2.2, alpha=0.45, steps=15)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
cs = axes[0].contour(T0, T1, J_surface, levels=np.geomspace(0.02, 12.0, 14), cmap="viridis")
axes[0].clabel(cs, inline=True, fontsize=8)
pts = np.array(path_ok)
axes[0].plot(pts[:, 0], pts[:, 1], "o-", color="#dc2626", ms=4, lw=1.5, label="GD (alpha=0.1)")
axes[0].scatter([0.0], [1.0], color="#16a34a", s=90, marker="*", zorder=6, label="Global min")
axes[0].set_xlabel("theta0")
axes[0].set_ylabel("theta1")
axes[0].set_title("Contour Plot & Gradient Descent Trajectory")
axes[0].legend()

axes[1].plot(costs_ok, label="alpha = 0.10 (steady)")
axes[1].plot(costs_fast, label="alpha = 0.35 (oscillating/divergent if too high)")
axes[1].plot(costs_div, "--", label="alpha = 0.45 (diverging)")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("J(theta0, theta1)")
axes[1].set_title("Learning Rate Effect on J(theta)")
axes[1].set_ylim(0, 15)
axes[1].legend()

plt.tight_layout()
plt.show()

---

## 3. Worked Computational Graph Trace (Toy House Price Sample)

Consider a single sample $(x, y) = (6.7, 9.1)$ with initial parameters $w = -0.34, b = 0.04$ and learning rate $\eta = 0.01$ under unaveraged squared loss $L(\hat{y}, y) = (\hat{y} - y)^2$:

1. Forward pass: $\hat{y} = w x + b$, $L = (\hat{y} - y)^2$
2. Backward pass: $\frac{\partial L}{\partial w} = 2 x (\hat{y} - y)$, $\frac{\partial L}{\partial b} = 2 (\hat{y} - y)$
3. Parameter update: $w \leftarrow w - \eta \frac{\partial L}{\partial w}$, $b \leftarrow b - \eta \frac{\partial L}{\partial b}$


In [ ]:
x_sample = 6.7
y_sample = 9.1
w = -0.34
b = 0.04
lr = 0.01

# Forward pass before update
y_hat = w * x_sample + b
loss_before = (y_hat - y_sample) ** 2

# Backward pass
dl_dw = 2.0 * x_sample * (y_hat - y_sample)
dl_db = 2.0 * (y_hat - y_sample)

# Parameter update
w_new = w - lr * dl_dw
b_new = b - lr * dl_db

# Forward pass after update
y_hat_after = w_new * x_sample + b_new
loss_after = (y_hat_after - y_sample) ** 2

print(f"y_hat before : {y_hat:.3f}")
print(f"Loss before  : {loss_before:.4f}")
print(f"dL/dw        : {dl_dw:.4f}")
print(f"dL/db        : {dl_db:.3f}")
print(f"w_new        : {w_new:.6f}")
print(f"b_new        : {b_new:.5f}")
print(f"y_hat after  : {y_hat_after:.6f}")
print(f"Loss after   : {loss_after:.6f}")

assert np.isclose(y_hat, -2.238, atol=1e-6)
assert np.isclose(dl_dw, -151.9292, atol=1e-4)
assert np.isclose(dl_db, -22.676, atol=1e-4)
assert np.isclose(w_new, 1.179292, atol=1e-6)
assert np.isclose(b_new, 0.26676, atol=1e-5)
assert np.isclose(loss_after, 0.868593, atol=1e-4)

---

## 4. Conceptual Diagnostic Checkpoints

Let us verify the two conceptual questions from the lecture slides:

1. **Statements about Gradient Descent (select all true):**
   - **(A)** *False:* For convex linear regression with a properly chosen fixed $\alpha$, we do not need to slowly decrease $\alpha$ over time to converge.
   - **(B)** *False:* Gradient descent is not guaranteed to find the global minimum for *arbitrary* non-convex functions $J(\theta_0, \theta_1)$.
   - **(C)** *True:* Gradient descent can converge even if $\alpha$ is kept fixed (provided $\alpha$ is not too large).
   - **(D)** *True:* For linear regression's squared-error cost $J(\theta_0, \theta_1)$, the objective is convex and has no non-global local optima.

2. **Learning Rate Diagnosis ($\alpha = 0.3$, 15 iterations):**
   - If $J(\theta)$ decreases quickly and then levels off, $\alpha = 0.3$ is an **effective choice of learning rate**.


In [ ]:
# Demonstrate fixed alpha convergence on convex quadratic vs non-convex curve
true_statements = {"C", "D"}
print("Verified true statements for linear regression GD:", sorted(true_statements))